# CTB ProSiT reproduction (Google Colab)
You should be able to upload this complete folder to `MyDrive`, open this notebook in Colab, and select **Run all**.

## 1. Mount Drive and create an isolated Python environment
The isolated environment prevents Colab's preinstalled NumPy and pandas versions from changing the run.

In [ ]:
from google.colab import drive
drive.mount('/content/MyDrive')

import os
import subprocess
import sys
from pathlib import Path

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        'Select a Python 3.11 Colab runtime (Runtime -> Change Runtime Type -> Runtime Version 2025.07)  and run the notebook again.'
    )

# ------------------------------------------------------------
# Find the handover folder automatically
# ------------------------------------------------------------

DRIVE_ROOT = Path('/content/MyDrive/MyDrive/CTB_ProSiT_reproduction')

matches = list(DRIVE_ROOT.rglob('reproduce.py'))

if len(matches) == 0:
    raise FileNotFoundError(
        f'Could not find reproduce.py anywhere inside {DRIVE_ROOT}'
    )

if len(matches) > 1:
    print('Multiple reproduce.py files found:')
    for match in matches:
        print(' -', match)

    raise RuntimeError(
        'Multiple handover folders found. '
        'Please specify ROOT manually.'
    )

ROOT = matches[0].parent

print('Handover folder found:')
print(ROOT)

os.chdir(ROOT)

# ------------------------------------------------------------
# Create isolated environment
# ------------------------------------------------------------

VENV = Path('/content/ctb_prosit_env')
PYTHON = VENV / 'bin' / 'python'

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'virtualenv'],
    check=True
)

if not PYTHON.is_file():
    subprocess.run(
        [sys.executable, '-m', 'virtualenv', str(VENV)],
        check=True
    )

subprocess.run(
    [
        str(PYTHON),
        '-m',
        'pip',
        'install',
        '-q',
        '-r',
        str(ROOT / 'requirements.txt')
    ],
    check=True
)

ENV = os.environ.copy()
ENV['MPLBACKEND'] = 'Agg'
ENV['PYTHONUNBUFFERED'] = '1'

def run_isolated(source):
    subprocess.run(
        [str(PYTHON), '-c', source],
        cwd=ROOT,
        env=ENV,
        check=True
    )

print()
print('Isolated environment:', PYTHON)

## 2. Load and inspect PNML, JSON and PKL models
JSON is the readable ProSiT export; PKL preserves the exact calibrated runtime object used for the thesis.

In [ ]:
run_isolated("""
import reproduce

print("JSON models loaded with SimulatorParameters.from_json():")
print(
    reproduce.inspect_models(
        reproduce.load_json_models()
    ).to_string(index=False)
)

print()
print("Exact thesis PKL models:")

models = reproduce.load_pickle_models()

print(
    reproduce.inspect_models(
        models
    ).to_string(index=False)
)

print()
print("Intervention checks:")

print(
    reproduce.check_model_changes(
        models
    ).to_string(index=False)
)
""")

## 3. Reproduce and compare the thesis scenario results
The default executes 10 matched seeds × 3 models × 17,892 cases and can take approximately 30 minutes.

In [ ]:
run_isolated("""
import reproduce

RUN_FULL = True

print("Starting CTB reproduction...")
print(f"Full reproduction: {RUN_FULL}")
print()

output_dir = reproduce.run(full=RUN_FULL)

print("Fresh result tables:")
print(output_dir)

if RUN_FULL:
    print()
    print("Comparing fresh results with thesis reference tables...")
    print()

    comparison = reproduce.compare(output_dir)

    print(comparison.to_string(index=False))

    print()
    print("FULL REPRODUCTION PASSED")
""")

## Reading the result
`outputs/full/scenario_paired_delta_summary.csv` contains the paired scientific effects. `outputs/full/comparison_with_expected_results.csv` is the technical reproducibility check; every `values_match` entry must be `True`. The notebook reproduces simulation from frozen models, not discovery from the confidential CTB event log.